In [ ]:
import torch
device= "cpu"
from previous_chapters import *
checkpoint= torch.load("model_and_optimizer.pth", map_location=device)
model= GPTModel(GPT_CONFIG_124M)
model.load_state_dict(checkpoint["model_state_dict"])
optimizer= torch.optim.AdamW(model.parameters(), lr= 5e-4, weight_decay = 0.1)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
model.train();

In [ ]:
GPT_CONFIG_124M ={
    "vocab_size" : 50257,
    "context_length" : 256,
    "emb_dim" : 768,
    "n_heads":12,
    "n_layers":12,
    "drop_rate":0.1,
    "qkv_bias": False
}

In [ ]:
def train_model_simple(model, train_loader, val_loader,
                       optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    train_losses , val_losses, track_tokens_seen = [],[],[]
    tokens_seen, global_step= 0,-1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss= calc_loss_batch(
                input_batch, target_batch, model, device
            )
            loss.backward()
            optimizer.step()

            tokens_seen += input_batch.numel()   #??
            global_step +=1

            if global_step % eval_freq ==0:
                train_loss, val_loss = evaluate_model(   #??
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)

                print(f"{epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f},"
                      f"val loss{val_loss:.3f}"
                      )
    
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )

    
    return train_losses, val_losses, track_tokens_seen

In [ ]:
torch.manual_seed(123)
from previous_chapters import *
# model= GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr= 0.0004, weight_decay= 0.1
)
num_epochs=10
train_losses, val_losses, tokens_seen= train_model_simple(
    model,train_loader, val_loader, optimizer, device,
    num_epochs= num_epochs, eval_freq=5, eval_iter=5,
    start_context= "Hare krishna", tokenizer= tokenizer
)

NameError: name 'train_loader' is not defined